In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set legacy Keras mode BEFORE any TensorFlow/Keras imports
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras  # Use tf_keras for TFMOT compatibility
from tf_keras import layers
from tf_keras.preprocessing.image import ImageDataGenerator
import tensorflow_model_optimization as tfmot
import tensorflow.lite as tflite

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("CPUs:", tf.config.list_physical_devices("CPU"))



TF version: 2.16.1
GPUs: []
CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [49]:
DATASET_PATH = "D:/waste_project/waste-classification-system-II/resized_dataset"

img_size = (96, 96)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale"
)

class_names = train_ds.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print("Classes:", class_names)
print("Number of classes:", num_classes)


Found 8407 files belonging to 9 classes.
Using 6726 files for training.
Found 8407 files belonging to 9 classes.
Using 1681 files for validation.
Classes: ['cardboard', 'e-waste', 'glass', 'metal', 'organic', 'paper', 'plastic', 'textile', 'trash']
Number of classes: 9


In [50]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)


In [53]:
model = models.Sequential([
    layers.Input(shape=(96, 96, 1)),
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.4),

    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [54]:
history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


Epoch 1/10
211/211 [==============================] - 46s 190ms/step - loss: 2.0733 - accuracy: 0.1925 - val_loss: 2.0264 - val_accuracy: 0.2463
Epoch 2/10
211/211 [==============================] - 41s 185ms/step - loss: 1.9182 - accuracy: 0.3003 - val_loss: 1.8505 - val_accuracy: 0.3432
Epoch 3/10
211/211 [==============================] - 40s 181ms/step - loss: 1.8070 - accuracy: 0.3512 - val_loss: 1.7825 - val_accuracy: 0.3641
Epoch 4/10
211/211 [==============================] - 40s 183ms/step - loss: 1.7394 - accuracy: 0.3820 - val_loss: 1.7201 - val_accuracy: 0.3902
Epoch 5/10
211/211 [==============================] - 85s 395ms/step - loss: 1.7073 - accuracy: 0.3797 - val_loss: 1.7139 - val_accuracy: 0.4045
Epoch 6/10
211/211 [==============================] - 41s 186ms/step - loss: 1.6413 - accuracy: 0.4169 - val_loss: 1.6733 - val_accuracy: 0.4081
Epoch 7/10
211/211 [==============================] - 44s 200ms/step - loss: 1.6059 - accuracy: 0.4301 - val_loss: 1.6081 - val_ac

In [32]:
inputs = keras.Input(shape=(96, 96, 3))

x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="mobilenetv3_small_96x96_waste")
model.summary()


Model: "mobilenetv3_small_96x96_waste"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_9 (InputLayer)        [(None, 96, 96, 3)]       0         
                                                                 
 MobilenetV3small (Function  (None, 3, 3, 432)         583160    
 al)                                                             
                                                                 
 global_average_pooling2d_4  (None, 432)               0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dropout_4 (Dropout)         (None, 432)               0         
                                                                 
 dense_4 (Dense)             (None, 9)                 3897      
                                                                 
Total params: 587057 (2.24 MB)
Traina

In [34]:
base_model.trainable = True

for layer in base_model.layers[:40]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


Epoch 1/10
211/211 [==============================] - 66s 168ms/step - loss: 1.8453 - accuracy: 0.3411 - val_loss: 1.3845 - val_accuracy: 0.5193
Epoch 2/10
211/211 [==============================] - 28s 125ms/step - loss: 1.2993 - accuracy: 0.5467 - val_loss: 1.2033 - val_accuracy: 0.5747
Epoch 3/10
211/211 [==============================] - 30s 134ms/step - loss: 1.1060 - accuracy: 0.6149 - val_loss: 1.0602 - val_accuracy: 0.6359
Epoch 4/10
211/211 [==============================] - 30s 132ms/step - loss: 0.9788 - accuracy: 0.6664 - val_loss: 0.9371 - val_accuracy: 0.6710
Epoch 5/10
211/211 [==============================] - 31s 140ms/step - loss: 0.8903 - accuracy: 0.6894 - val_loss: 0.8959 - val_accuracy: 0.6889
Epoch 6/10
211/211 [==============================] - 33s 148ms/step - loss: 0.8226 - accuracy: 0.7117 - val_loss: 0.8652 - val_accuracy: 0.6859
Epoch 7/10
211/211 [==============================] - 31s 134ms/step - loss: 0.7452 - accuracy: 0.7371 - val_loss: 0.8348 - val_ac

In [8]:
base_model.trainable = True

for layer in base_model.layers[:40]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


Epoch 1/10
211/211 [==============================] - 39s 102ms/step - loss: 0.9986 - accuracy: 0.6496 - val_loss: 0.8985 - val_accuracy: 0.6847
Epoch 2/10
211/211 [==============================] - 28s 76ms/step - loss: 0.8929 - accuracy: 0.6894 - val_loss: 0.8742 - val_accuracy: 0.6871
Epoch 3/10
211/211 [==============================] - 17s 72ms/step - loss: 0.8322 - accuracy: 0.7034 - val_loss: 0.8090 - val_accuracy: 0.7192
Epoch 4/10
211/211 [==============================] - 16s 66ms/step - loss: 0.7646 - accuracy: 0.7324 - val_loss: 0.7895 - val_accuracy: 0.7281
Epoch 5/10
211/211 [==============================] - 16s 67ms/step - loss: 0.7100 - accuracy: 0.7472 - val_loss: 0.7672 - val_accuracy: 0.7353
Epoch 6/10
211/211 [==============================] - 16s 66ms/step - loss: 0.6741 - accuracy: 0.7542 - val_loss: 0.7908 - val_accuracy: 0.7281
Epoch 7/10
211/211 [==============================] - 16s 65ms/step - loss: 0.6294 - accuracy: 0.7752 - val_loss: 0.7527 - val_accuracy

In [35]:
model.save("mobilenetv3_small_96x96_alpha075_float.keras")


In [36]:
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

def apply_pruning_to_dense(layer):
    if isinstance(layer, keras.layers.Dense):
        return prune_low_magnitude(
            layer,
            pruning_schedule=tfmot.sparsity.keras.PolynomialDecay(
                initial_sparsity=0.0,
                final_sparsity=0.5,
                begin_step=0,
                end_step=1000
            )
        )
    return layer

pruned_model = keras.models.clone_model(
    model,
    clone_function=apply_pruning_to_dense
)

pruned_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_history = pruned_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)


Epoch 1/5
211/211 [==============================] - 63s 160ms/step - loss: 0.5619 - accuracy: 0.8000 - val_loss: 0.7743 - val_accuracy: 0.7347
Epoch 2/5
211/211 [==============================] - 29s 126ms/step - loss: 0.5278 - accuracy: 0.8149 - val_loss: 0.7001 - val_accuracy: 0.7662
Epoch 3/5
211/211 [==============================] - 29s 127ms/step - loss: 0.4906 - accuracy: 0.8284 - val_loss: 0.7257 - val_accuracy: 0.7632
Epoch 4/5
211/211 [==============================] - 29s 128ms/step - loss: 0.4629 - accuracy: 0.8378 - val_loss: 0.7397 - val_accuracy: 0.7555
Epoch 5/5
211/211 [==============================] - 31s 139ms/step - loss: 0.4456 - accuracy: 0.8420 - val_loss: 0.7317 - val_accuracy: 0.7513


In [37]:
history.history["accuracy"][-1], history.history["val_accuracy"][-1]


(0.6297948360443115, 0.6335514783859253)

In [38]:
final_train_acc = history_ft.history["accuracy"][-1]
final_val_acc = history_ft.history["val_accuracy"][-1]
print(final_train_acc, final_val_acc)


0.7873921990394592 0.7364664077758789


In [39]:
import tensorflow_model_optimization as tfmot

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

def apply_pruning_to_dense(layer):
    if isinstance(layer, keras.layers.Dense):
        return prune_low_magnitude(layer)
    return layer

pruned_model = keras.models.clone_model(
    model,
    clone_function=apply_pruning_to_dense
)


In [15]:
pruned_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)


Epoch 1/5
211/211 [==============================] - 37s 86ms/step - loss: 0.5330 - accuracy: 0.8095 - val_loss: 0.6994 - val_accuracy: 0.7585
Epoch 2/5
211/211 [==============================] - 17s 69ms/step - loss: 0.4992 - accuracy: 0.8244 - val_loss: 0.7075 - val_accuracy: 0.7609
Epoch 3/5
211/211 [==============================] - 16s 69ms/step - loss: 0.4961 - accuracy: 0.8202 - val_loss: 0.7067 - val_accuracy: 0.7686
Epoch 4/5
211/211 [==============================] - 16s 68ms/step - loss: 0.4539 - accuracy: 0.8347 - val_loss: 0.7212 - val_accuracy: 0.7626
Epoch 5/5
211/211 [==============================] - 19s 80ms/step - loss: 0.4383 - accuracy: 0.8454 - val_loss: 0.6803 - val_accuracy: 0.7728


In [40]:
model_for_export = tfmot.sparsity.keras.strip_pruning(pruned_model)
model_for_export.save("mobilenetv3_small_96x96_alpha075_pruned.keras")


In [41]:
def representative_data_gen():
    for images, _ in train_ds.take(100):
        yield [tf.cast(images, tf.float32)]


### quantization

In [42]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

with open("mobilenetv3_small_96x96_alpha075_int8.tflite", "wb") as f:
    f.write(tflite_int8_model)

print("Final INT8 size (KB):", len(tflite_int8_model) / 1024)


INFO:tensorflow:Assets written to: C:\Users\Asghar\AppData\Local\Temp\tmpknq6hwx0\assets


INFO:tensorflow:Assets written to: C:\Users\Asghar\AppData\Local\Temp\tmpknq6hwx0\assets
C:\Users\Asghar\anaconda3\envs\waste_env\lib\site-packages\tensorflow\lite\python\convert.py:964: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Final INT8 size (KB): 799.1953125


In [35]:
def representative_data_gen():
    for images, _ in train_ds.take(100):
        yield [tf.cast(images, tf.float32)]


In [32]:
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.16.1


In [25]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot

print("TF version:", tf.__version__)
print("TFLite converter:", hasattr(tf.lite, "TFLiteConverter"))
print("INT8 ops:", tf.lite.OpsSet.TFLITE_BUILTINS_INT8)
print("Pruning available:", hasattr(tfmot.sparsity.keras, "prune_low_magnitude"))


TF version: 2.16.1
TFLite converter: True
INT8 ops: TFLITE_BUILTINS_INT8
Pruning available: True


In [36]:
import tensorflow_model_optimization as tfmot
model = tfmot.sparsity.keras.strip_pruning(model)


ValueError: ('Expected model to be a `keras.Model` instance but got: ', <Functional name=mobilenetv3_small_96x96_waste, built=True>)

In [39]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf

In [40]:


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("mobilenetv3_small_96x96_int8.tflite", "wb") as f:
    f.write(tflite_model)

AttributeError: 'Functional' object has no attribute '_get_save_spec'

In [55]:
import os
print(os.listdir("D:/waste_project/waste-classification-system-II/resized_dataset"))

['cardboard', 'e-waste', 'glass', 'metal', 'organic', 'paper', 'plastic', 'textile', 'trash']
